# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [1]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

../../../../../../assets/03-wine-quality/inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [2]:
# Leemos el CSV fila a fila; cada fila es un diccionario {columna: valor}
with DATA_PATH.open(newline="", encoding="utf-8") as source:
    rows = list(csv.DictReader(source))  # guardamos todas las filas en una lista

row = rows[0]  # la primera fila (red-001)

print("Número de filas:", len(rows))
print("Columnas:", list(row.keys()))
print("Primera fila:", row)

Número de filas: 5
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Primera fila: {'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?
2. ¿Cuántas características numéricas hay?
3. ¿Qué columna extra añadirías para probar el contrato?
4. ¿Qué valor de `ph` sería claramente inválido?

**Respuestas:**

1. `sample_id`. Identifica la muestra, pero el modelo no la usa para predecir.
2. 11 características numéricas.
3. `unexpected_field` (cualquier columna que no esté en el contrato).
4. `ph = 99`. El pH del vino está entre 2.5 y 4.5, así que 99 es imposible.

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [3]:
class WineQualityRequestPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid")  # prohíbe columnas que no estén aquí

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)


# Separamos sample_id: no es una característica del modelo
model_row = {key: value for key, value in row.items() if key != "sample_id"}
request = WineQualityRequestPydantic.model_validate(model_row)  # valida y convierte a float
print(request)

fixed_acidity=7.4 volatile_acidity=0.7 citric_acid=0.0 residual_sugar=1.9 chlorides=0.076 free_sulfur_dioxide=11.0 total_sulfur_dioxide=34.0 density=0.9978 ph=3.51 sulphates=0.56 alcohol=9.4


### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

**Errores esperados (antes de ejecutar):**

- Fila válida: ningún error.
- `unexpected_field`: error `extra_forbidden` (columna no permitida).
- `ph = 99`: error `less_than_equal` en `ph` (fuera de rango).

In [4]:
valid_row = dict(model_row)                                     # caso 1: fila válida
extra_row = {**model_row, "unexpected_field": "not allowed"}    # caso 2: columna extra
bad_ph_row = {**model_row, "ph": "99"}                          # caso 3: ph fuera de rango

cases = {"fila válida": valid_row, "columna extra": extra_row, "ph fuera de rango": bad_ph_row}

for name, case in cases.items():
    try:
        WineQualityRequestPydantic.model_validate(case)
        print(f"{name}: OK, el contrato la acepta")
    except ValidationError as error:
        print(f"{name}: RECHAZADA -> {error.errors()[0]['type']} en {error.errors()[0]['loc']}")

fila válida: OK, el contrato la acepta
columna extra: RECHAZADA -> extra_forbidden en ('unexpected_field',)
ph fuera de rango: RECHAZADA -> less_than_equal en ('ph',)


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [5]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float


valid = WineQualityRequestDataclass(**{k: float(v) for k, v in model_row.items()})
print("Instancia válida:", valid)

# Con ph=99 la dataclass NO se queja: no comprueba rangos
bad = WineQualityRequestDataclass(**{**{k: float(v) for k, v in model_row.items()}, "ph": 99.0})
print("ph=99 aceptado sin error:", bad.ph)

# Y tampoco convierte tipos: guarda el texto "7.4" tal cual
text = WineQualityRequestDataclass(**model_row)
print("Tipo de fixed_acidity si viene del CSV:", type(text.fixed_acidity))

Instancia válida: WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=3.51, sulphates=0.56, alcohol=9.4)
ph=99 aceptado sin error: 99.0
Tipo de fixed_acidity si viene del CSV: <class 'str'>


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en __________ porque __________. Usaría `dataclass` en __________ porque __________.

**Respuesta:**

> Usaría **Pydantic** en **la entrada del sistema (el CSV o una API)** porque **valida tipos y rangos, convierte el texto a número y rechaza columnas extra**. Usaría **`dataclass`** en **el interior del módulo, con datos ya validados (por ejemplo, el vector de características)** porque **es ligera y no necesita volver a validar lo que ya se comprobó**.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [6]:
# Orden exacto con el que se entrenó el modelo (igual que el CSV, sin sample_id)
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

# Recorremos los nombres EN ORDEN y sacamos cada valor de la petición validada
vector = [getattr(request, name) for name in FEATURE_NAMES]
print(len(vector), "valores:", vector)

11 valores: [7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
- ¿El clasificador detectaría necesariamente el error?
- ¿Cómo evitarías que cada script definiera un orden distinto?

**Respuestas:**

- Sí, sigue teniendo 11 valores, porque solo cambian de sitio.
- No. El clasificador solo ve números y no sabe qué significa cada posición, así que predice sin avisar (y puede predecir mal).
- Definiendo `FEATURE_NAMES` **una sola vez** en `preprocess.py` y comprobando, al cargar el modelo, que coincide con los `feature_names` guardados en el artefacto `.joblib`.

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [7]:
from typing import Literal


class WineQualityPredictionPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid")

    sample_id: str = Field(min_length=1)                                   # ¿qué muestra?
    quality_band: Literal["needs_review", "acceptable", "excellent"]       # ¿qué categoría?
    confidence: float = Field(ge=0, le=1)                                  # ¿con qué confianza?
    model_version: str = Field(min_length=1)                               # ¿qué modelo?
    preprocessing_version: str = Field(min_length=1)                       # ¿qué preprocesado?


example = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="acceptable",
    confidence=0.82,
    model_version="wine-quality-rf-v1",
    preprocessing_version="wine-red-features-v1",
)
print(example.model_dump())

{'sample_id': 'red-001', 'quality_band': 'acceptable', 'confidence': 0.82, 'model_version': 'wine-quality-rf-v1', 'preprocessing_version': 'wine-red-features-v1'}


## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

**Recorrido anotado:**

1. **CSV**: `csv.DictReader` lee cada fila como un diccionario de textos.
2. **Contrato**: `WineQualityRequest` valida tipos, rangos y columnas.
3. **Preprocesado**: `preprocess_wine_request` construye el vector en el orden de `FEATURE_NAMES`.
4. **Modelo cargado**: se abre el `.joblib` y se comprueba que sus `feature_names` coinciden.
5. **Predicción**: `predict` y `predict_proba` dan la categoría y la confianza, que se validan con `WineQualityPrediction`.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.

## Lienzo de la práctica 3.1 (resumen de entrega)

### 1. Entrenamiento, artefacto y datos nuevos

| Pieza | Qué es | Quién la usa |
|---|---|---|
| Datos de entrenamiento | CSV histórico de vinos tintos **con** la etiqueta de calidad | El docente, una sola vez, para entrenar |
| Artefacto | `models/wine_quality_classifier.joblib`: modelo ya entrenado + `feature_names` + `model_version` | Nuestro módulo: solo lo carga, no lo reentrena |
| Datos nuevos | `inference_samples.csv`: 5 muestras **sin** etiqueta | Nuestro módulo: les pide una predicción |

### 2. Tabla del contrato de entrada (11 campos)

| # | Campo | Tipo | Unidad | Límite razonable |
|---|---|---|---|---|
| 1 | `fixed_acidity` | float | g ácido tartárico / dm³ | 0 – 20 |
| 2 | `volatile_acidity` | float | g ácido acético / dm³ | 0 – 2 |
| 3 | `citric_acid` | float | g / dm³ | 0 – 2 |
| 4 | `residual_sugar` | float | g / dm³ | 0 – 20 |
| 5 | `chlorides` | float | g cloruro sódico / dm³ | 0 – 1 |
| 6 | `free_sulfur_dioxide` | float | mg / dm³ | 0 – 100 |
| 7 | `total_sulfur_dioxide` | float | mg / dm³ | 0 – 300 |
| 8 | `density` | float | g / cm³ | 0.98 – 1.01 |
| 9 | `ph` | float | sin unidad (escala pH) | 2.5 – 4.5 |
| 10 | `sulphates` | float | g sulfato potásico / dm³ | 0 – 3 |
| 11 | `alcohol` | float | % vol | 5 – 20 |

`sample_id` (texto, no vacío) identifica la fila, pero **no** entra en el modelo. No se permite ninguna columna adicional.

### 3. Ejemplos de fila

**Válida (red-001):**

```text
sample_id=red-001, fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9,
chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978,
ph=3.51, sulphates=0.56, alcohol=9.4
```

**Inválida (columna no permitida):** la misma fila con `unexpected_field=not allowed` produce un error `extra_forbidden` y el script se detiene.

**Otros casos de fallo:** `ph=99` (fuera de rango), `alcohol=invalid` (no es un número), falta `alcohol` (campo obligatorio) y `sample_id` vacío.

### 4. Orden exacto del vector

`fixed_acidity, volatile_acidity, citric_acid, residual_sugar, chlorides, free_sulfur_dioxide, total_sulfur_dioxide, density, ph, sulphates, alcohol`

### 5. Criterios de aceptación del script de la clase 2

1. Una fila válida genera una predicción con `sample_id`, `quality_band`, `confidence`, `model_version` y `preprocessing_version`.
2. Una columna extra, un campo que falte o un valor fuera de rango detienen el script con un error claro (código de salida 2) y **no se escribe ningún CSV de salida**, ni siquiera parcial.
3. Si los `feature_names` del artefacto no coinciden con `FEATURE_NAMES`, el script se niega a predecir.

### Plan de implementación (clase 2)

- `contracts.py`: `WineQualityRequest` (11 campos, rangos, `extra="forbid"`) y `WineQualityPrediction`.
- `preprocess.py`: `FEATURE_NAMES`, `PREPROCESSING_VERSION` y `preprocess_wine_request`, que devuelve el vector ordenado.
- `inference.py`: cargar el `.joblib`, comprobar el clasificador y `feature_names`, y predecir categoría y confianza.
- `predict_file.py`: leer el CSV, validar e inferir **todas** las filas en memoria y escribir el CSV solo al final.